# Module System — Interview Traps

These are the questions that separate developers who **use** the import system
from those who **understand** it.

In [ ]:
import sys
from pathlib import Path

_demo = Path.cwd() / 'demo_packages'
if not _demo.exists():
    _demo = Path.cwd() / 'modules_and_imports' / 'demo_packages'
assert _demo.exists(), f'demo_packages not found — expected: {_demo}'

if str(_demo) not in sys.path:
    sys.path.insert(0, str(_demo))
print(f'demo_packages on path: {sys.path[0]}')

### Trap 1 — `from X import y` vs `import X` — which survives mutation?

When you write `from pkg_a.core import greet`, Python creates a **new local binding**  
named `greet` that points to the function object. It is NOT a live reference to `pkg_a.core.greet`.

If someone mutates `pkg_a.core.greet` (replaces it) after your import, your local `greet`  
still points to the original function — you get **stale data**.

```python
# ❌ stale after mutation
from pkg_a.core import greet
pkg_a.core.greet = lambda name: 'patched!'   # rebinds the module attribute
greet('Alice')   # still calls the ORIGINAL — local binding was not updated

# ✅ always current
import pkg_a.core
pkg_a.core.greet('Alice')   # looks up greet on the module at call time
```

**When this matters in production:** mocking with `unittest.mock.patch`.  
`patch('pkg_a.core.greet')` patches the module attribute.  
Code that uses `from pkg_a.core import greet` won't see the mock.  
Code that uses `pkg_a.core.greet(...)` will.

In [ ]:
import pkg_a.core
from pkg_a.core import greet   # local binding created here

original = pkg_a.core.greet    # save original
print(f'Before patch: greet("Alice") = {greet("Alice")!r}')

# Simulate what mock.patch does — replaces the module attribute
pkg_a.core.greet = lambda name: f'[PATCHED] {name}'

print(f'After patch (local binding): greet("Alice") = {greet("Alice")!r}')
print(f'After patch (module attr):   pkg_a.core.greet("Alice") = {pkg_a.core.greet("Alice")!r}')

# Restore
pkg_a.core.greet = original

### Trap 2 — Module-level code runs at import time

Any code at the top level of a module (outside functions/classes) runs when the module is **imported**,
not when a function in it is called.

```python
# expensive_module.py
result = very_expensive_computation()   # ← runs at import time!

# Caller:
import expensive_module   # ← computation happens HERE, not when you use result
```

**Interview angle:**  
"Why is your application slow to start?" — Often: module-level code (DB connections,
file reads, heavy computation) running at import time.

**The fix:** lazy initialization — move expensive work inside a function or use `__init__`.

In [ ]:
import time

# Simulating a module with expensive import-time side effects
class SlowModule:
    pass

# Pretend this is module-level code in a .py file:
start = time.perf_counter()
time.sleep(0.1)   # simulate heavy computation at module level
SlowModule.data = [i ** 2 for i in range(10_000)]
elapsed = time.perf_counter() - start

print(f'Module-level code ran at import time: {elapsed:.3f}s')
print(f'SlowModule.data[:3] = {SlowModule.data[:3]}')
print()
print('Fix: wrap in a function and call lazily:')
print('  _cache = None')
print('  def get_data():')
print('      global _cache')
print('      if _cache is None: _cache = compute()')
print('      return _cache')

### Trap 3 — sys.modules can be mutated

`sys.modules` is a regular dict — anyone can modify it.  
`del sys.modules['X']` removes the cached module. The next `import X` re-executes it from disk.

This is used in tests to force a fresh import of a module (simulating a clean state).  
In production code, mutating `sys.modules` is almost always a bug.

**What happens after deletion:**
- The module object still exists if anyone holds a reference to it
- The next `import X` creates a NEW module object — existing references still point to the OLD one
- `import X` after deletion: re-runs `__init__.py` prints, resets module-level state

In [ ]:
import lifecycle_demo.once as once_ref   # capture reference

old_count = once_ref.get_count()         # call on current module
print(f'Before del: get_count() = {old_count}')
print(f'Module id before: {id(once_ref)}')

# Delete from cache
if 'lifecycle_demo.once' in sys.modules:
    del sys.modules['lifecycle_demo.once']

# Re-import — this re-executes once.py (init print fires again)
print()
print('--- re-importing after del sys.modules ---')
import lifecycle_demo.once as once_fresh
print()

print(f'Module id after:  {id(once_fresh)}')
print(f'Same object: {once_ref is once_fresh}')   # False — new object
print(f'once_ref still works (old object): get_count() = {once_ref.get_count()}')
print(f'once_fresh (new object):           get_count() = {once_fresh.get_count()}')

### Trap 4 — `__name__ == '__main__'` fires only when run directly

Every module has `__name__` set to its fully qualified name when **imported**.  
When run directly (`python myfile.py`), `__name__` is set to `'__main__'`.

```python
# mymodule.py
def run():
    print('running')

if __name__ == '__main__':
    run()   # ← this ONLY runs when: python mymodule.py
            #   NOT when: import mymodule
```

**The trap:** code outside `if __name__ == '__main__':` runs at import time.  
If you put side effects (print, network calls, file writes) at module level,  
they run even when the module is imported by someone else.

In [ ]:
import pkg_a.core as core_mod

print(f'pkg_a.core.__name__ = {core_mod.__name__!r}')
print(f'Is it __main__? {core_mod.__name__ == "__main__"}')
print()
print(f'This notebook.__name__ = {__name__!r}')   # __main__ in notebook context
print(f'Is this notebook __main__? {__name__ == "__main__"}')

### Trap 5 — Star import pollutes namespace and can shadow

`from module import *` dumps all names from `__all__` (or all non-underscore names  
if `__all__` is absent) into your current namespace.

**Problems:**
1. Silent shadowing: a name from `module` overwrites a name you already defined
2. No traceability: `grep` for `Counter` in the file doesn't show where it came from
3. Brittle: if the module adds a new name that matches one of yours, things break silently

**Rule:** never use `from X import *` in production code. Only acceptable in REPLs and `__init__.py`  
(and even there, be explicit about `__all__`).

In [ ]:
# Demonstrating silent shadowing

Counter = 42   # local variable
print(f'Counter before star import: {Counter}')

from pkg_a import *   # Counter from pkg_a.__all__ OVERWRITES local Counter

print(f'Counter after star import:  {Counter}')
print(f'  (was 42, now {type(Counter).__name__} — silently shadowed!)')
print()
print(f'pkg_a.__all__ = {pkg_a.__all__!r}')

### Trap 6 — `importlib.reload` doesn't update existing references

`reload(module)` re-executes the module file and updates `module.__dict__` in place.  
But any names you already extracted (`from module import name`) still point to the OLD object.

```python
from mymodule import greet   # local binding to old greet function

importlib.reload(mymodule)   # mymodule.greet is now a NEW function object

greet('Alice')               # still calls OLD greet — local binding unchanged!
mymodule.greet('Alice')      # calls NEW greet — attribute lookup happens at call time
```

**Why:** `reload` replaces names in `module.__dict__`, but your local `greet` is a separate  
variable in your scope. It was bound once and doesn't auto-update when the module does.

In [ ]:
import importlib
import pkg_a.core
from pkg_a.core import greet   # captures current greet object

print(f'greet id before reload: {id(greet)}')
print(f'module greet id before: {id(pkg_a.core.greet)}')
print(f'Same object: {greet is pkg_a.core.greet}')

importlib.reload(pkg_a.core)   # re-runs core.py — new greet function created

print()
print(f'greet id after reload:  {id(greet)}')          # unchanged
print(f'module greet id after:  {id(pkg_a.core.greet)}')  # changed
print(f'Same object: {greet is pkg_a.core.greet}')    # False — stale reference!

### Trap 7 — Import inside a function: when it's intentional vs a code smell

**When it's the right pattern:**
1. Circular import resolution (see notebook 03)
2. Optional heavyweight dependency (only import if the feature is used)
3. Plugin/extension loading inside factory functions

**When it's a smell:**
1. Hiding a slow import that should be at the top-level
2. Avoiding a circular import that should be resolved structurally
3. Putting it inside a loop (redundant — sys.modules caches after first call, but it's misleading)

**The nuance:** `import X` inside a function is NOT slow after the first call —  
Step 1 hits `sys.modules` immediately. But it's unusual enough to warrant a comment.

In [ ]:
import time

# After first import, subsequent calls are essentially free
def heavy_feature():
    import json   # already in sys.modules — this is a dict lookup, not file I/O
    return json.dumps({'status': 'ok'})

# Benchmark: first call vs subsequent calls
times = []
for _ in range(1000):
    start = time.perf_counter()
    heavy_feature()
    times.append(time.perf_counter() - start)

print(f'First call:     {times[0]*1_000_000:.2f} µs')
print(f'Average after:  {sum(times[1:]) / len(times[1:]) * 1_000_000:.2f} µs')
print(f'Overhead of deferred import: near zero after first call')

### Trap 8 — the entry-point script can be initialized twice

Run `python app.py` and Python sets `sys.modules['__main__']` to a module object for `app.py`,
then executes the file with `__name__ == '__main__'`.

The key insight: that module is cached under the key `'__main__'` — **not** under the file's
real module name (`'app'`). If another module in the same program later does `import app`,
Python finds no matching key, so it loads `app.py` from disk **again**, creates a **second,
distinct module object**, and executes the entire file **again** — top to bottom.

```python
# app.py
print(f'module-level code — __name__ = {__name__!r}')

def main():
    print('main() — entry-point-only logic')

if __name__ == '__main__':
    main()
```

Run directly: both lines print, cached as `sys.modules['__main__']`.
Imported elsewhere as `app`: the `print` at module level fires *again* (second execution,
second object, cached as `sys.modules['app']`) — but `main()` never runs, because `__name__`
is `'app'` there, not `'__main__'`.

**Why this matters:** any module-level side effect in your entry-point file (DB connections,
argument parsing, logging setup) runs **twice** if that file is ever imported by its real name
elsewhere — once as `__main__`, once as itself. This is exactly the failure the
`if __name__ == '__main__':` guard exists to contain.

In [ ]:
import subprocess

app_path = _demo / 'entry_point_demo' / 'app.py'

# Genuine "python app.py" — a fresh process, __name__ == '__main__' for real,
# sys.modules['__main__'] is set inside THAT process (invisible to us here).
result = subprocess.run(
    [sys.executable, str(app_path)],
    capture_output=True, text=True, check=True,
)
print('--- ran as a script: python app.py ---')
print(result.stdout)

In [ ]:
# Now import the SAME file, by its real module name, in THIS process.
# Python has no 'entry_point_demo.app' key in sys.modules yet — it loads
# and executes app.py from scratch. main() does NOT run: __name__ is
# 'entry_point_demo.app' here, not '__main__'.
print('--- importing the same file as entry_point_demo.app ---')
import entry_point_demo.app as app_mod

print()
print(f'app_mod.__name__ = {app_mod.__name__!r}')
in_cache = 'entry_point_demo.app' in sys.modules
print(f"'entry_point_demo.app' in sys.modules: {in_cache}")
print('This import is a SEPARATE module object from whatever ran in the subprocess —')
print('two executions of the same source file, two distinct module objects.')

**What just happened:**

| | Run as script (`python app.py`) | Imported as `entry_point_demo.app` |
|---|---|---|
| `__name__` | `'__main__'` | `'entry_point_demo.app'` |
| sys.modules key | `'__main__'` | `'entry_point_demo.app'` |
| Module-level `print` | fires | fires **again** |
| `main()` | runs | does **not** run |

Same source file. Two full executions. Two distinct module objects. This is why
production code puts side effects behind `if __name__ == '__main__':` — it's the only
thing standing between "runs once" and "runs once per way this file gets loaded."

### Summary — interview cheat sheet

| Question | Key answer |
|---|---|
| What is a module at runtime? | A `types.ModuleType` object with `__dict__`, `__name__`, `__file__` |
| What is sys.modules? | A dict cache; modules are initialized once and stored here |
| When does a module run? | When first imported (Steps 1-5); never again unless `reload()` |
| Package vs module? | Package has `__path__`; both are `ModuleType` |
| What does `import pkg_a.core` bind? | Only `pkg_a` in namespace, not `core` |
| Why do circular imports fail? | `from X import name` requires `name` at import time; X may not be done |
| Fix for circular import? | Deferred import, import module not symbol, or restructure |
| `from X import y` vs `import X`? | `from` creates a local copy — stale after mutation; `import` always fresh |
| When does `__name__ == '__main__'`? | Only when run directly, never when imported |
| Does reload update `from X import y`? | No — local binding stays stale; only `X.y` gets the new object |
| Can the entry-point script run twice? | Yes — once as `'__main__'`, again if imported by its real name; module-level code re-executes each time |